# Mental Health Dataset — French EDA Pipeline




In [155]:
# ── Standard library ──────────────────────────────────────────────────────────
import os
import re
import string
import warnings
from collections import Counter
from itertools import combinations

# ── External libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

warnings.filterwarnings("ignore")
print(" All imports OK")


 All imports OK


##  Configuration

In [156]:
df=pd.read_csv("french_data.csv")
df.head()   
df.info()

FileNotFoundError: [Errno 2] No such file or directory: 'french_data.csv'

In [ ]:
df["text_length"] = pd.to_numeric(df["text_length"], errors="coerce")
df["age"].unique()
df["age"] = df["age"].astype("category")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6258 entries, 0 to 6257
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   text             6258 non-null   object  
 1   word_count       6258 non-null   int64   
 2   language         6258 non-null   object  
 3   category         6258 non-null   object  
 4   age              6258 non-null   category
 5   education_level  6258 non-null   object  
 6   formality        6258 non-null   object  
 7   context          6258 non-null   object  
 8   mental_state     6258 non-null   object  
 9   text_length      0 non-null      float64 
 10  length_category  6258 non-null   object  
dtypes: category(1), float64(1), int64(1), object(8)
memory usage: 495.4+ KB


In [ ]:
class Config:
    """
    Central configuration — change values here, everything else adapts.
    """

    # ── File paths ────────────────────────────────────────────────────────────
    CSV_PATH       = r"french_data.csv"          # path to your dataset
    STOPWORDS_FILE = r"french_stopwords.txt"     # path to French stopwords file
    OUTPUT_DIR     = r"MyResults"                   # folder for saved plots

    # ── Column names ─────────────────────────────────────────────────────────
    LANGUAGE_COL   = "language"
    LANGUAGE_VALUE = "French"
    TEXT_COL       = "text"
    LABEL_COL      = "mental_state"

    # ── Plot settings ─────────────────────────────────────────────────────────
    BG      = "#F9F9F9"    # background color for all plots
    DPI     = 150          # resolution when saving figures
    PALETTE = "Set2"       # seaborn color palette

    # ── Analysis settings ─────────────────────────────────────────────────────
    TOP_N_WORDS = 20       # top N most frequent words per label
    TOP_N_COOC  = 20       # top N word co-occurrence pairs per label

cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
print(f"Config ready — output folder: '{cfg.OUTPUT_DIR}'")


Config ready — output folder: 'MyResults'


## 🎨 Plot Helper

In [ ]:
class PlotHelper:
    """
    Centralizes styling and saving of all matplotlib figures.
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        plt.rcParams.update({
            "figure.facecolor" : cfg.BG,
            "axes.facecolor"   : cfg.BG,
            "axes.spines.top"  : False,
            "axes.spines.right": False,
            "font.size"        : 11,
        })

    def save(self, filename: str) -> str:
        """Save the current figure and return its path."""
        path = os.path.join(self.cfg.OUTPUT_DIR, filename)
        plt.tight_layout()
        plt.savefig(path, dpi=self.cfg.DPI, bbox_inches="tight")
        plt.close()
        print(f"  [SAVED] {filename}")
        return path

    @staticmethod
    def safe_name(text: str) -> str:
        """Convert a string to a safe filename (removes forbidden chars)."""
        return re.sub(r'[\\/*?"<>|]+', "_", str(text)).strip()

helper = PlotHelper(cfg)
print("PlotHelper ready")


PlotHelper ready


##  Data Loader

In [ ]:
class DataLoader:
    """Loads the CSV and filters to French rows only."""

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def load(self) -> pd.DataFrame:
        df = pd.read_csv(self.cfg.CSV_PATH)
        print(f"[DataLoader] Total rows loaded       : {len(df)}")

        # Case-insensitive filter on language column
        mask = (
            df[self.cfg.LANGUAGE_COL].str.strip().str.lower()
            == self.cfg.LANGUAGE_VALUE.lower()
        )
        df = df[mask].copy().reset_index(drop=True)
        print(f"[DataLoader] French rows kept         : {len(df)}")
        return df


##  Text Cleaner

In [ ]:
class TextCleaner:
    """
    Cleans the text column and adds helper columns:
      - text_clean      : lowercased, no URLs / punctuation / emojis / emoticons
      - text_nostop     : text_clean without stopwords
      - word_count      : words in cleaned text
      - char_count      : characters in original text
      - punct_count     : punctuation marks in original
      - emoji_count     : emoji characters in original
      - emoticon_count  : text emoticons in original
    """

    # ── Regex patterns ────────────────────────────────────────────────────────
    EMOJI_RE = re.compile(
        "[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF"
        "\U00002700-\U000027BF\U000024C2-\U0001F251]+",
        flags=re.UNICODE,
    )
    EMOTICON_RE = re.compile(
        r"(:-\)|:\)|:-\(|:\(|;-\)|;\)|:D|XD|<3|:P|:-P|:O|B\))"
    )
    URL_RE     = re.compile(r"https?://\S+|www\.\S+")
    PUNCT_CHARS = set(string.punctuation + "«»–—…")

    def __init__(self, stopwords_file: str):
        """Load stopwords from file and merge with custom mental-health stopwords."""
        try:
            with open(stopwords_file, "r", encoding="utf-8") as f:
                file_stopwords = {line.strip().lower() for line in f if line.strip()}
        except FileNotFoundError:
            print(f" Stopwords file '{stopwords_file}' not found — using custom only.")
            file_stopwords = set()

        custom_stopwords = {
            # greetings
            "bonjour", "salut", "merci", "svp", "sil", "plait",
            # consultation words
            "question", "reponse", "docteur", "medecin", "aide", "conseil",
            # vague words
            "chose", "probleme", "situation", "sentiment",
        }
        self.FRENCH_STOPWORDS = file_stopwords | custom_stopwords
        print(f"[TextCleaner] Stopwords loaded        : {len(self.FRENCH_STOPWORDS)}")

    # ── Counters ──────────────────────────────────────────────────────────────
    def _count_emojis(self, text: str)    -> int: return len(self.EMOJI_RE.findall(str(text)))
    def _count_emoticons(self, text: str) -> int: return len(self.EMOTICON_RE.findall(str(text)))
    def _count_punct(self, text: str)     -> int: return sum(1 for c in str(text) if c in self.PUNCT_CHARS)

    # ── Cleaners ──────────────────────────────────────────────────────────────
    def _clean(self, text: str) -> str:
        text = str(text)
        text = self.URL_RE.sub("", text)
        text = self.EMOJI_RE.sub("", text)
        text = self.EMOTICON_RE.sub("", text)
        text = text.translate(str.maketrans("", "", string.punctuation + "«»–—…"))
        return re.sub(r"\s+", " ", text).strip().lower()

    def _remove_stopwords(self, text: str) -> str:
        return " ".join(
            w for w in text.split()
            if w not in self.FRENCH_STOPWORDS and len(w) > 2
        )

    # ── Public API ────────────────────────────────────────────────────────────
    def fit_transform(self, df: pd.DataFrame, text_col: str) -> pd.DataFrame:
        df  = df.copy()
        raw = df[text_col].astype(str)

        # Count BEFORE cleaning (on original text)
        df["emoji_count"]    = raw.apply(self._count_emojis)
        df["emoticon_count"] = raw.apply(self._count_emoticons)
        df["punct_count"]    = raw.apply(self._count_punct)
        df["char_count"]     = raw.apply(len)

        # Clean text
        df["text_clean"]  = raw.apply(self._clean)
        df["text_nostop"] = df["text_clean"].apply(self._remove_stopwords)
        df["word_count"]  = df["text_clean"].apply(lambda x: len(x.split()))

        # Remove duplicates and empty rows
        before = len(df)
        df.drop_duplicates(subset="text_clean", inplace=True)
        df = df[df["text_clean"].str.strip() != ""].reset_index(drop=True)
        print(f"[TextCleaner] Removed {before - len(df)} duplicate/empty rows → {len(df)} remain")
        return df


##  Load & Clean the Data

In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df_raw = DataLoader(cfg).load()

# ── Clean ─────────────────────────────────────────────────────────────────────
cleaner  = TextCleaner(stopwords_file=cfg.STOPWORDS_FILE)
df       = cleaner.fit_transform(df_raw, cfg.TEXT_COL)

# ── Save cleaned CSV ──────────────────────────────────────────────────────────
clean_path = os.path.join(cfg.OUTPUT_DIR, "french_cleaned.csv")
df.to_csv(clean_path, index=False, encoding="utf-8-sig")
print(f"\n Cleaned CSV saved → {clean_path}")
df.head(3)


[DataLoader] Total rows loaded       : 6258
[DataLoader] French rows kept         : 6258
 Stopwords file 'french_stopwords.txt' not found — using custom only.
[TextCleaner] Stopwords loaded        : 16
[TextCleaner] Removed 239 duplicate/empty rows → 6019 remain

 Cleaned CSV saved → MyResults\french_cleaned.csv


,text,word_count,language,category,age,education_level,formality,context,mental_state,text_length,length_category,emoji_count,emoticon_count,punct_count,char_count,text_clean,text_nostop
0,Ma valeur transcende mes moments sombres,6,French,Self-Worth,young adult (20-29),professional,highly formal academic,text message to friend,Healthy,between 3 and 8 words,3-8 words,0,0,0,40,ma valeur transcende mes moments sombres,valeur transcende mes moments sombres
1,Je mérite de vivre pleinement chaque jour,7,French,Self-Worth,elderly (60+),primary education,formal,diary entry,Healthy,between 3 and 8 words,3-8 words,0,0,0,41,je mérite de vivre pleinement chaque jour,mérite vivre pleinement chaque jour
2,"Je mérite de vivre, pour vrai!",6,French,Self-Worth,adult (30-45),primary education,very informal with slang,text message to friend,Healthy,between 3 and 8 words,3-8 words,0,0,2,30,je mérite de vivre pour vrai,mérite vivre pour vrai


##   Dataset Size

In [ ]:
class DatasetSize:
    """Req 0 — General statistics about the dataset (no plot needed)."""

    def __init__(self, cfg: Config):
        self.cfg = cfg

    def analyse(self, df: pd.DataFrame) -> dict:
        stats = {
            "Total rows (French)"  : len(df),
            "Unique texts"         : df["text_clean"].nunique(),
            "Columns"              : len(df.columns),
            "Labels"               : df[self.cfg.LABEL_COL].nunique(),
            "Avg word count"       : round(df["word_count"].mean(), 2),
            "Max word count"       : df["word_count"].max(),
            "Min word count"       : df["word_count"].min(),
            "Avg char count"       : round(df["char_count"].mean(), 2),
            "Missing values total" : int(df.isnull().sum().sum()),
        }
        print("\n[DatasetSize]")
        for k, v in stats.items():
            print(f"  {k:<28}: {v}")
        return stats

stats = DatasetSize(cfg).analyse(df)



[DatasetSize]
  Total rows (French)         : 6019
  Unique texts                : 6019
  Columns                     : 17
  Labels                      : 2
  Avg word count              : 25.13
  Max word count              : 88
  Min word count              : 4
  Avg char count              : 154.78
  Missing values total        : 0


## 1️⃣  Label Distribution

In [ ]:
class LabelDistribution:
    """Req 1 — Bar chart + pie chart of label counts."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        counts = df[self.cfg.LABEL_COL].value_counts()

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Label Distribution — mental_state", fontsize=14, fontweight="bold")

        # Bar chart
        sns.barplot(x=counts.values, y=counts.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[0])
        axes[0].set_xlabel("Count")
        axes[0].set_title("Count per Label")
        for bar, val in zip(axes[0].patches, counts.values):
            axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
                         str(val), va="center", fontsize=9)

        # Pie chart
        axes[1].pie(counts.values, labels=counts.index,
                    autopct="%1.1f%%",
                    colors=sns.color_palette(self.cfg.PALETTE, len(counts)),
                    startangle=140)
        axes[1].set_title("Proportion per Label")

        return self.helper.save("01_label_distribution.png")

LabelDistribution(cfg, helper).analyse(df)


  [SAVED] 01_label_distribution.png


'MyResults\\01_label_distribution.png'

## 2️⃣  Text Length Analysis

In [ ]:
class TextLengthAnalysis:
    """Req 2 — Histogram of word count + boxplot of char count by label."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Text Length Analysis", fontsize=14, fontweight="bold")

        # Histogram + KDE of word count
        axes[0].hist(df["word_count"], bins=30, color="#4C9BE8",
                     edgecolor="white", density=True, alpha=0.7, label="Word Count")
        df["word_count"].plot.kde(ax=axes[0], color="darkblue", linewidth=2, label="KDE")
        axes[0].axvline(df["word_count"].mean(), color="red", linestyle="--",
                        label=f"Mean={df['word_count'].mean():.1f}")
        axes[0].set_title("Word Count Distribution")
        axes[0].set_xlabel("Words per text")
        axes[0].set_ylabel("Density")
        axes[0].legend()

        # Boxplot of char count by label
        order = (df.groupby(self.cfg.LABEL_COL)["char_count"]
                   .median().sort_values(ascending=False).index)
        sns.boxplot(data=df, x=self.cfg.LABEL_COL, y="char_count",
                    order=order, palette=self.cfg.PALETTE, ax=axes[1])
        axes[1].set_title("Char Count by Label")
        axes[1].set_xlabel("")
        plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

        return self.helper.save("02_text_length.png")

TextLengthAnalysis(cfg, helper).analyse(df)


  [SAVED] 02_text_length.png


'MyResults\\02_text_length.png'

## 3️⃣  Punctuation Usage

In [ ]:
class PunctuationAnalysis:
    """Req 3 — Histogram of punct per text + avg punct per label bar chart."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle("Punctuation Usage", fontsize=14, fontweight="bold")

        # Histogram
        axes[0].hist(df["punct_count"], bins=25, color="#E87B4C", edgecolor="white")
        axes[0].axvline(df["punct_count"].mean(), color="red", linestyle="--",
                        label=f"Mean={df['punct_count'].mean():.1f}")
        axes[0].set_title("Punct Count Distribution")
        axes[0].set_xlabel("Punctuation marks per text")
        axes[0].legend()

        # Avg punctuation per label
        avg = (df.groupby(self.cfg.LABEL_COL)["punct_count"]
                  .mean().sort_values(ascending=False))
        sns.barplot(x=avg.values, y=avg.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1])
        axes[1].set_title("Avg Punct Count by Label")
        axes[1].set_xlabel("Avg count")

        return self.helper.save("03_punctuation.png")

PunctuationAnalysis(cfg, helper).analyse(df)


  [SAVED] 03_punctuation.png


'MyResults\\03_punctuation.png'

## 4️⃣  Word Clouds per Label

In [ ]:
class WordCloudAnalysis:
    """Req 4 — One word cloud per label (using text_nostop)."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> list:
        labels = df[self.cfg.LABEL_COL].unique()
        n      = len(labels)
        cols   = min(3, n)
        rows   = (n + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 4))
        axes = np.array(axes).flatten()
        fig.suptitle("Word Clouds per Label", fontsize=15, fontweight="bold")

        cmaps = ["Blues", "Reds", "Greens", "Purples", "Oranges", "YlOrBr"]

        for i, label in enumerate(labels):
            text = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"])
            if not text.strip():
                axes[i].axis("off")
                continue

            wc = WordCloud(
                width=600, height=350,
                background_color="white",
                colormap=cmaps[i % len(cmaps)],
                max_words=100,
                collocations=False,
            ).generate(text)

            axes[i].imshow(wc, interpolation="bilinear")
            axes[i].axis("off")
            axes[i].set_title(str(label), fontsize=12, fontweight="bold")

        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        path = self.helper.save("04_wordclouds_per_label.png")
        return [path]

WordCloudAnalysis(cfg, helper).analyse(df)


  [SAVED] 04_wordclouds_per_label.png


['MyResults\\04_wordclouds_per_label.png']

## 5️⃣  Word Co-occurrence per Label

In [ ]:
class CoOccurrenceAnalysis:
    """Req 5 — Top word pairs that appear together per label (horizontal bar chart)."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def _cooccurrence(self, texts, top_n):
        co = Counter()
        for sentence in texts:
            words = list(set(sentence.split()))
            for pair in combinations(sorted(words), 2):
                co[pair] += 1
        return co.most_common(top_n)

    def analyse(self, df: pd.DataFrame) -> list:
        paths  = []
        labels = df[self.cfg.LABEL_COL].unique()

        for label in labels:
            texts = df[df[self.cfg.LABEL_COL] == label]["text_nostop"]
            pairs = self._cooccurrence(texts, self.cfg.TOP_N_COOC)
            if not pairs:
                continue

            pair_labels = [f"{a} & {b}" for (a, b), _ in pairs]
            counts      = [c for _, c in pairs]

            fig, ax = plt.subplots(figsize=(10, 6))
            sns.barplot(x=counts, y=pair_labels, palette="mako", ax=ax)
            ax.set_title(f"Top Word Co-occurrences — {label}", fontsize=13, fontweight="bold")
            ax.set_xlabel("Co-occurrence count")

            fname = f"05_cooccurrence_{self.helper.safe_name(label)}.png"
            paths.append(self.helper.save(fname))

        return paths

CoOccurrenceAnalysis(cfg, helper).analyse(df)


  [SAVED] 05_cooccurrence_Healthy.png
  [SAVED] 05_cooccurrence_Unhealthy.png


['MyResults\\05_cooccurrence_Healthy.png',
 'MyResults\\05_cooccurrence_Unhealthy.png']

## 6️⃣  Common Words per Label

In [ ]:
class CommonWordsAnalysis:
    """Req 6 — Top N most frequent words per label (horizontal bar chart)."""

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> list:
        labels = df[self.cfg.LABEL_COL].unique()
        cols   = min(2, len(labels))
        rows   = (len(labels) + cols - 1) // cols

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 7, rows * 5))
        axes = np.array(axes).flatten()
        fig.suptitle(f"Top {self.cfg.TOP_N_WORDS} Common Words per Label",
                     fontsize=14, fontweight="bold")

        for i, label in enumerate(labels):
            words = " ".join(df[df[self.cfg.LABEL_COL] == label]["text_nostop"]).split()
            freq  = Counter(words).most_common(self.cfg.TOP_N_WORDS)
            if not freq:
                axes[i].axis("off")
                continue
            w, c = zip(*freq)
            sns.barplot(x=list(c), y=list(w), palette="rocket", ax=axes[i])
            axes[i].set_title(str(label), fontsize=11, fontweight="bold")
            axes[i].set_xlabel("Frequency")

        for j in range(i + 1, len(axes)):
            axes[j].axis("off")

        path = self.helper.save("06_common_words_per_label.png")
        return [path]

CommonWordsAnalysis(cfg, helper).analyse(df)


  [SAVED] 06_common_words_per_label.png


['MyResults\\06_common_words_per_label.png']

##   Emoji & Emoticon Analysis

In [ ]:
class EmojiEmoticonAnalysis:
    """
    Req 7 & 8 — Emoji and emoticon usage analysis.
    Produces:
      - Bar chart: avg emoji count per label
      - Bar chart: avg emoticon count per label
      - Histogram: distribution of emoji counts
      - Histogram: distribution of emoticon counts
    """

    def __init__(self, cfg: Config, helper: PlotHelper):
        self.cfg    = cfg
        self.helper = helper

    def analyse(self, df: pd.DataFrame) -> str:
        fig, axes = plt.subplots(2, 2, figsize=(13, 10))
        fig.suptitle("Emoji & Emoticon Analysis", fontsize=14, fontweight="bold")

        # ── Top-left: emoji count distribution (histogram) ────────────────────
        axes[0, 0].hist(df["emoji_count"], bins=20, color="#F4A460", edgecolor="white")
        axes[0, 0].axvline(df["emoji_count"].mean(), color="red", linestyle="--",
                           label=f"Mean={df['emoji_count'].mean():.2f}")
        axes[0, 0].set_title("Emoji Count Distribution")
        axes[0, 0].set_xlabel("Emojis per text")
        axes[0, 0].legend()

        # ── Top-right: emoticon count distribution (histogram) ────────────────
        axes[0, 1].hist(df["emoticon_count"], bins=20, color="#87CEEB", edgecolor="white")
        axes[0, 1].axvline(df["emoticon_count"].mean(), color="red", linestyle="--",
                           label=f"Mean={df['emoticon_count'].mean():.2f}")
        axes[0, 1].set_title("Emoticon Count Distribution")
        axes[0, 1].set_xlabel("Emoticons per text")
        axes[0, 1].legend()

        # ── Bottom-left: avg emoji count per label (bar chart) ────────────────
        avg_emoji = (df.groupby(self.cfg.LABEL_COL)["emoji_count"]
                       .mean().sort_values(ascending=False))
        sns.barplot(x=avg_emoji.values, y=avg_emoji.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1, 0])
        axes[1, 0].set_title("Avg Emoji Count by Label")
        axes[1, 0].set_xlabel("Avg emoji count")

        # ── Bottom-right: avg emoticon count per label (bar chart) ────────────
        avg_emot = (df.groupby(self.cfg.LABEL_COL)["emoticon_count"]
                      .mean().sort_values(ascending=False))
        sns.barplot(x=avg_emot.values, y=avg_emot.index.astype(str),
                    palette=self.cfg.PALETTE, ax=axes[1, 1])
        axes[1, 1].set_title("Avg Emoticon Count by Label")
        axes[1, 1].set_xlabel("Avg emoticon count")

        return self.helper.save("07_08_emoji_emoticon.png")

EmojiEmoticonAnalysis(cfg, helper).analyse(df)


  [SAVED] 07_08_emoji_emoticon.png


'MyResults\\07_08_emoji_emoticon.png'

##  Pipeline Summary

In [ ]:
print("=" * 55)
print("  FRENCH EDA PIPELINE — COMPLETE")
print("=" * 55)

saved = [f for f in os.listdir(cfg.OUTPUT_DIR) if f.endswith(".png")]
print(f"\n📊 {len(saved)} plots saved to '{cfg.OUTPUT_DIR}/':")
for f in sorted(saved):
    print(f"   • {f}")

print(f"\n Cleaned CSV : {cfg.OUTPUT_DIR}/french_cleaned.csv")
print("\n All done!")


  FRENCH EDA PIPELINE — COMPLETE

📊 8 plots saved to 'MyResults/':
   • 01_label_distribution.png
   • 02_text_length.png
   • 03_punctuation.png
   • 04_wordclouds_per_label.png
   • 05_cooccurrence_Healthy.png
   • 05_cooccurrence_Unhealthy.png
   • 06_common_words_per_label.png
   • 07_08_emoji_emoticon.png

📄 Cleaned CSV : MyResults/french_cleaned.csv

✅ All done!
